
# Upper reward timing around handover

This notebook runs one deterministic upper-agent window and instruments the local simulator ticks inside that window.

The example uses `fixed_center_embb_left_right`: all eMBB UEs start on center gNB 1, then the upper action applies a negative eMBB bias from gNB 1 toward both neighbors. The important question is whether the transient PRB spike at handover time enters the upper reward.

What to look for:

- Local steps `0..3` are the post-handover settle phase.
- The radio measurement window opens after local step `4` starts.
- Reward load uses only the measured window average after settle, not the transient settle samples.
- Load is useful PRBs divided by physical gNB PRBs.


In [1]:

from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = next(
    path for path in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]
    if (path / "global_ppo_3gnb_env.py").exists()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from global_ppo_3gnb_env import GlobalPPO3GNBEnv
from upper_agent_training_scenarios import CENTER_LEFT_RIGHT_GNB_CONFIGS


def load_balance_cost(loads):
    loads = np.asarray(loads, dtype=float)
    slice_means = loads.mean(axis=0, keepdims=True)
    return float(np.sum((loads - slice_means) ** 2))


def run_demo():
    env = GlobalPPO3GNBEnv(
        seed=123,
        gnb_configs=CENTER_LEFT_RIGHT_GNB_CONFIGS,
        scenario_mode="curriculum",
        training_scenarios="fixed_center_embb_left_right",
        upper_window_seconds=1.0,
        local_steps_per_global=10,
        radio_substeps=2,
        terminal_reward_only=False,
        max_handovers_per_local_step=3,
        a3_handover_cooldown_s=2.0,
        a3_min_residence_s=2.0,
        safe_admission_enabled=True,
        warmup_steps=0,
        post_handover_settle_steps=4,
    )
    try:
        _obs, reset_info = env.reset(seed=123)
        timeline = []
        step_counter = {"n": 0}
        phase = {"name": "settle"}
        measurement_open_after = []
        original_step = env.base_env.step
        original_begin = env.base_env.begin_radio_measurement_window

        def current_embb_loads():
            matrix = env._load_matrix()
            return {
                "g0_eMBB": float(matrix[0, 0]),
                "g1_eMBB": float(matrix[1, 0]),
                "g2_eMBB": float(matrix[2, 0]),
            }

        def counted_step(action):
            before_events = len(env.base_env.get_handover_events())
            result = original_step(action)
            after_events = list(env.base_env.get_handover_events())
            new_events = after_events[before_events:]
            row = {
                "local_step": step_counter["n"],
                "phase": phase["name"],
                "new_handovers": len(new_events),
                "routes": ", ".join(
                    f"UE{event['ue_id']}:g{event['from_gnb']}->g{event['to_gnb']}"
                    for event in new_events
                ),
            }
            row.update(current_embb_loads())
            timeline.append(row)
            step_counter["n"] += 1
            return result

        def counted_begin():
            measurement_open_after.append(step_counter["n"])
            phase["name"] = "measure"
            return original_begin()

        env.base_env.step = counted_step
        env.base_env.begin_radio_measurement_window = counted_begin

        action = np.zeros(env.action_space.shape, dtype=np.float32)
        action.reshape(3, 2, 3)[1, :, 0] = -0.4
        _obs, reward, terminated, truncated, info = env.step(action)

        start_loads = np.asarray(info["load_matrix_start"], dtype=float)
        end_loads = np.asarray(info["load_matrix_end"], dtype=float)
        start_cost = load_balance_cost(start_loads)
        end_cost = load_balance_cost(end_loads)
        raw_improvement = start_cost - end_cost
        scaled_load_reward = float(np.clip(raw_improvement / max(start_cost, 0.05), -1.0, 1.0))
        reconstructed_reward = (
            info["reward_load_improvement"]
            + info["reward_saturation_improvement"]
            + info["reward_excess_load_improvement"]
            - info["global_action_penalty"]
            - info["global_negative_bias_penalty"]
        )

        return {
            "timeline": pd.DataFrame(timeline),
            "events": pd.DataFrame(env.base_env.get_handover_events()),
            "measurement_open_after": measurement_open_after,
            "reward": reward,
            "info": info,
            "start_loads": start_loads,
            "end_loads": end_loads,
            "start_cost": start_cost,
            "end_cost": end_cost,
            "raw_improvement": raw_improvement,
            "scaled_load_reward": scaled_load_reward,
            "reconstructed_reward": reconstructed_reward,
        }
    finally:
        env.close()


result = run_demo()
info = result["info"]

print("UPPER WINDOW SUMMARY")
print(f"post_handover_settle_steps: {info['post_handover_settle_steps']}")
print(f"radio_measurement_steps:     {info['radio_measurement_steps']}")
print(f"measurement window opened after local step count: {result['measurement_open_after']}")
print(f"load_measurement_mode:       {info['load_measurement_mode']}")
print(f"handover_count:              {info['handover_count']}")
print(f"reward returned to PPO:      {result['reward']:.6f}")
print()

print("LOCAL STEP TIMELINE")
print(result["timeline"].to_string(index=False))
print()

print("HANDOVER EVENTS")
cols = ["step", "ue_id", "slice_type", "from_gnb", "to_gnb", "offset_db", "safe_admission"]
print(result["events"][cols].to_string(index=False))
print()

print("LOAD MATRICES USED BY THE REWARD")
print("Rows are gNB0, gNB1, gNB2. Columns are eMBB, URLLC, mMTC.")
print("start_loads = previous window average / reset useful-PRB load")
print(np.array2string(result["start_loads"], precision=3))
print("end_loads = window-average useful-PRB load after settle")
print(np.array2string(result["end_loads"], precision=3))
print()

print("LOAD REWARD ARITHMETIC")
print(f"start_imbalance = sum((L_start - per_slice_mean)^2) = {result['start_cost']:.6f}")
print(f"end_imbalance   = sum((L_end   - per_slice_mean)^2) = {result['end_cost']:.6f}")
print(f"raw improvement = start_imbalance - end_imbalance   = {result['raw_improvement']:.6f}")
print(f"scaled load reward = clip(raw / max(start, 0.05), -1, 1) = {result['scaled_load_reward']:.6f}")
print()

print("FULL WINDOW REWARD")
print(f"load improvement reward:     {info['reward_load_improvement']:.6f}")
print(f"saturation improvement:      {info['reward_saturation_improvement']:.6f}")
print(f"excess-load improvement:     {info['reward_excess_load_improvement']:.6f}")
print(f"minus action penalty:         {info['global_action_penalty']:.6f}")
print(f"minus negative-bias penalty:  {info['global_negative_bias_penalty']:.6f}")
print(f"reconstructed reward:         {result['reconstructed_reward']:.6f}")
print(f"dense_window_reward in info:  {info['dense_window_reward']:.6f}")


UPPER WINDOW SUMMARY
post_handover_settle_steps: 4
radio_measurement_steps:     6
measurement window opened after local step count: [4]
load_measurement_mode:       window_average_useful_prbs
handover_count:              4
reward returned to PPO:      1.950569

LOCAL STEP TIMELINE
 local_step   phase  new_handovers                             routes  g0_eMBB  g1_eMBB  g2_eMBB
          0  settle              0                                        0.00     0.84     0.00
          1  settle              0                                        0.00     0.98     0.00
          2  settle              3 UE0:g1->g0, UE1:g1->g2, UE4:g1->g0     0.33     0.46     0.18
          3  settle              1                         UE5:g1->g2     0.07     0.29     0.08
          4 measure              0                                        0.10     0.29     0.08
          5 measure              0                                        0.07     0.29     0.12
          6 measure              0     


## Interpretation

In this run, all successful handovers happen during the settle phase. The measurement window opens only after four local steps have completed, so the reward does not average the transient PRB values from the handover ticks.

The `end_loads` matrix is produced by `get_window_average_slice_loads()`, which averages useful PRBs over the measurement samples. The upper reward then compares `start_loads` against this settled `end_loads` matrix.
